In [15]:
from typing import Callable, Tuple
import math



## Пункт 1 - составная формула левых прямоугольников

In [16]:
def composite_left(f: Callable[[float], float], a: float, b: float, n: int) -> float:
    h = (b - a) / n
    s = 0.0
    for i in range(n):
        x = a + i * h
        s += f(x)
    return h * s

## Пункт 2 - составная формула правых прямоугольников

In [17]:
def composite_right(f: Callable[[float], float], a: float, b: float, n: int) -> float:
    h = (b - a) / n
    s = 0.0
    for i in range(1, n + 1):
        x = a + i * h
        s += f(x)
    return h * s

## Пункт 3 - составная формула центральных (midpoint) прямоугольников

In [18]:
def composite_midpoint(f: Callable[[float], float], a: float, b: float, n: int) -> float:
    h = (b - a) / n
    s = 0.0
    for i in range(n):
        x = a + i * h + 0.5 * h
        s += f(x)
    return h * s

$$ S = \frac{1}{2}(a+b)h $$

## Пункт 4 - формула трапеций (составная)

In [19]:
def composite_trapezoid(f: Callable[[float], float], a: float, b: float, n: int) -> float:
    h = (b - a) / n
    s = 0.5 * f(a) + 0.5 * f(b)
    for i in range(1, n):
        x = a + i * h
        s += f(x)
    return h * s

строим параболы по 3 точкам

## Пункт 5 - формула Симпсона (составная)
требует чётного n - здесь автоматически корректируем

In [20]:
def composite_simpson(f: Callable[[float], float], a: float, b: float, n: int) -> float:
    if n % 2 == 1:
        n += 1
    h = (b - a) / n
    s = f(a) + f(b)
    for i in range(1, n):
        x = a + i * h
        s += (4 if i % 2 == 1 else 2) * f(x)
    return h * s / 3.0

## Пункт 6 - формула Уэддла (Weddle)
Требует, чтобы n было кратно 6; здесь корректируем n вверх при необходимости

In [21]:
def composite_weddle(f: Callable[[float], float], a: float, b: float, n: int) -> float:
    if n % 6 != 0:
        n += (6 - n % 6)
    h = (b - a) / n
    weights = [1, 5, 1, 6, 1, 5, 1]
    s = 0.0
    j = 0
    while j < n:
        for k in range(7):
            x = a + (j + k) * h
            s += weights[k] * f(x)
        j += 6
    return 3.0 * h * s / 10.0

In [22]:
def integrate_once(method: str, f: Callable[[float], float], a: float, b: float, n: int) -> Tuple[float, int]:
    if method == "left":
        return composite_left(f, a, b, n), n
    if method == "right":
        return composite_right(f, a, b, n), n
    if method == "midpoint":
        return composite_midpoint(f, a, b, n), n
    if method == "trapezoid":
        return composite_trapezoid(f, a, b, n), n
    if method == "simpson":
        if n % 2 == 1:
            n += 1
        return composite_simpson(f, a, b, n), n
    if method == "weddle":
        if n % 6 != 0:
            n += (6 - n % 6)
        return composite_weddle(f, a, b, n), n
    raise ValueError("unknown method")

In [23]:

def integrate_adaptive(method: str, f: Callable[[float], float], a: float, b: float, eps: float, n0: int = 2, max_iter: int = 25) -> Tuple[float, int]:
    n = n0
    if method == "simpson" and n % 2 == 1:
        n += 1
    if method == "weddle" and n % 6 != 0:
        n += (6 - n % 6)
    In, n_used = integrate_once(method, f, a, b, n)
    for _ in range(max_iter):
        target = n_used * 2
        if method == "simpson" and target % 2 == 1:
            target += 1
        if method == "weddle" and target % 6 != 0:
            target += (6 - target % 6)
        I2n, n2 = integrate_once(method, f, a, b, target)
        if abs(In - I2n) <= eps:
            return I2n, n2
        In = I2n
        n_used = n2
    return In, n_used


In [24]:

def f(x):
    return math.sin(x)


In [25]:
methods = ["left", "right", "midpoint", "trapezoid", "simpson", "weddle"]

for m in methods:
    result, n_used = integrate_adaptive(m, f, 0, math.pi, 1e-6)
    print(m, "->", result, " n =", n_used)


left -> 1.9999999019542845  n = 4096
right -> 1.9999999019542845  n = 4096
midpoint -> 2.0000001960914378  n = 2048
trapezoid -> 1.9999999019542845  n = 4096
simpson -> 2.0000000645300013  n = 64
weddle -> 1.9999999879498886  n = 24


Neuton-Cotes formulae. Ланграндж 1го порядка приводит к формуле трапеций. \
Ланграндж 2го порядка приводит к $\frac{1}{3}$ Simpsons Rule (3 интервала).\
Ланграндж 3го порядка приводит к $\frac{3}{8}$ правилу Симпсона для 8 интервалов. \
Трапеции и Симпсон - частные случаи Newton–Cotes

Формулы Ньютона–Котеса основаны на интерполяции функции
$f(x)$ многочленом Лагранжа степени $n$ на отрезке $[a,b]$:
$$
f(x) \approx P_n(x).
$$



Интеграл от функции заменяется интегралом от интерполяционного многочлена:
$$
\int_a^b f(x)\,dx \approx \int_a^b P_n(x)\,dx.
$$

| Степень интерполяции | Число узлов | Метод          |
| -------------------- | ----------- | -------------- |
| n = 1                | 2           | Трапеции       |
| n = 2                | 3           | Симпсон 1/3    |
| n = 3                | 4           | Симпсон 3/8    |
| n = 4                | 5           | Правило Буля   |
| n = 5                | 6           | Правило Уэддля |


## Пункты 7-12

In [26]:
from typing import Callable, Tuple, List
import math

# Newton-Cotes closed weights, совпадают с перечисленными в задании (n -> список c0..cn)
NC_WEIGHTS = {
    1: [1/2, 1/2],                                     # Пункт 7 (n=1)
    2: [1/6, 4/6, 1/6],                                # Пункт 8 (n=2)
    3: [1/8, 3/8, 3/8, 1/8],                           # Пункт 9 (n=3)
    4: [7/90, 32/90, 12/90, 32/90, 7/90],              # Пункт 10 (n=4)
    5: [19/288, 75/288, 50/288, 50/288, 75/288, 19/288],# Пункт 11 (n=5)
    6: [41/840, 216/840, 27/840, 272/840, 27/840, 216/840, 41/840], # Пункт 12 (n=6)
}


def newton_cotes_on_interval(f: Callable[[float], float], a: float, b: float, n: int) -> float:
    if n not in NC_WEIGHTS:
        raise ValueError("Newton-Cotes weights available for n=1..6 only")
    weights = NC_WEIGHTS[n]
    h = (b - a) / n
    s = 0.0
    for k, ck in enumerate(weights):
        xk = a + k * h
        s += ck * f(xk)
    return (b - a) / n * s




### Квадратурные формулы Ньютона–Котеса и Гаусса

#### Формулы Ньютона–Котеса

Квадратурная формула Ньютона–Котеса (закрытого типа) имеет вид

$$
I_{NC} = (b - a)\sum_{k=0}^{n} c_k\, f(x_k),
\qquad
x_k = a + k\frac{b-a}{n}.
$$

Коэффициенты $c_k$ определяются интегрированием базисных многочленов Лагранжа:

$$
c_k = \frac{1}{b-a}\int_a^b \ell_k(x)\,dx,
$$

где $\ell_k(x)$ — базисные многочлены Лагранжа,
построенные по равномерной сетке узлов
$x_0, x_1, \dots, x_n$ на отрезке $[a,b]$.

---

#### Формулы Гаусса–Лежандра

Квадратурная формула Гаусса с $n$ узлами имеет вид

$$
I_G = \frac{b-a}{2}\sum_{k=1}^{n} c_k\,
f\!\left(
\frac{a+b}{2} + \frac{b-a}{2} t_k
\right).
$$

Здесь:
- $t_k$ — корни полинома Лежандра $P_n(t)$ на интервале $[-1,1]$;
- $c_k$ — соответствующие весовые коэффициенты;
- замена переменной переносит формулу с $[-1,1]$ на $[a,b]$.


NC — узлы равномерные (включая концы отрезка), веса получаются интегрированием базисных многочленов Лагранжа; это семейство правил (трапеция, Симпсон, Уэддл и т.д.).

Гаусс — узлы выбирают оптимально (неравномерно) — корни полинома Лежандра; веса подбирают так, чтобы правило было максимально точным: n-узловая формула Гаусса точно интегрирует все полиномы степени ≤ 2n−1.

In [27]:

GAUSS = {
    1: {"t": [0.0], "c": [2.0]},  # Пункт 13
    2: {"t": [-0.5773502691896257, 0.5773502691896257], "c": [1.0, 1.0]},  # Пункт 14
    3: {"t": [-0.7745966692414834, 0.0, 0.7745966692414834], "c": [5.0 / 9.0, 8.0 / 9.0, 5.0 / 9.0]},  # Пункт 15
    4: {"t": [-0.8611363115940526, -0.3399810435848563, 0.3399810435848563, 0.8611363115940526],
        "c": [0.34785484513745385, 0.6521451548625461, 0.6521451548625461, 0.34785484513745385]}  # Пункт 16
}

#13-16
def gauss_on_interval(f: Callable[[float], float], a: float, b: float, n: int) -> float:
    if n not in GAUSS:
        raise ValueError("Gauss implemented for n=1..4 only")
    data = GAUSS[n]
    t = data["t"]
    c = data["c"]
    mid = 0.5 * (a + b)
    half = 0.5 * (b - a)
    s = 0.0
    for tk, ck in zip(t, c):
        xk = mid + half * tk
        s += ck * f(xk)
    return half * s



In [28]:
#применить выбранную локальную формулу к равным подотрезкам
def apply_quadrature_on_equal_subintervals(
    quad_func: Callable[[Callable[[float], float], float, float, int], float],
    f: Callable[[float], float],
    a: float,
    b: float,
    n: int,
    parts: int
) -> float:
    #7-16
    total = 0.0
    width = (b - a) / parts
    for j in range(parts):
        sub_a = a + j * width
        sub_b = sub_a + width
        total += quad_func(f, sub_a, sub_b, n)
    return total

def adaptive_halving_scheme(
    family: str,
    f: Callable[[float], float],
    a: float,
    b: float,
    n: int,
    eps: float,
    max_halvings: int = 20
) -> Tuple[float, int]:
    if family == "nc":
        quad = newton_cotes_on_interval
    elif family == "gauss":
        quad = gauss_on_interval
    else:
        raise ValueError("family must be 'nc' or 'gauss'")

    parts = 1
    I_prev = apply_quadrature_on_equal_subintervals(quad, f, a, b, n, parts)
    for level in range(1, max_halvings + 1):
        parts *= 2
        I_next = apply_quadrature_on_equal_subintervals(quad, f, a, b, n, parts)
        if abs(I_prev - I_next) <= eps:
            return I_next, parts
        I_prev = I_next
    return I_prev, parts

## 7-12

In [29]:
import math

def f(x): return math.sin(x)

res, parts = adaptive_halving_scheme("nc", f, 0.0, math.pi, n=2, eps=1e-6, max_halvings=20)
print("Newton–Cotes n=2 result:", res, "parts:", parts, "error:", abs(res - 2.0))


NC n=2 result: 1.0000000322650016 parts: 32 error: 0.9999999677349984


## 13-16

In [30]:
res, parts = adaptive_halving_scheme("gauss", f, 0.0, math.pi, n=3, eps=1e-9, max_halvings=12)
print("Gauss n=3 result:", res, "parts:", parts, "error:", abs(res - 2.0))


Gauss n=3 result: 2.000000000000889 parts: 32 error: 8.890665981198254e-13
